# Ecommify — Unidad 4: Análisis de planes de ejecución con EXPLAIN ANALYZE

**Curso:** Maestría en Arquitectura de Software  
**Proyecto:** Ecommify — Plataforma de e-commerce  
**Actividad:** Etapa 1 — Investigación. Análisis de planes de ejecución  

---

## Objetivo

Identificar cuellos de botella en las consultas críticas del sistema Ecommify mediante
el análisis de planes de ejecución con `EXPLAIN` y `EXPLAIN ANALYZE`, documentando
métricas de costo, tiempo y tipo de scan para priorizar optimizaciones.

## Consultas analizadas

| ID | Consulta | Tipo |
|---|---|---|
| Q11 | Pedidos por cliente con total pagado | OLTP |
| Q12 | Detalle completo de orden (4 JOINs) | OLTP |
| Q13 | Tiempo promedio de entrega por estado | OLAP |
| Q16 | Score promedio de reviews por categoría | OLAP |
| Q5  | Filtro JSONB con operador @> | Tipo avanzado |
| Q6  | Filtro JSONB con cast numérico | Tipo avanzado |
| Q9  | Búsqueda fuzzy con pg_trgm | Extensión |

---
## Paso 1 — Conexión a Supabase

In [21]:
!pip install psycopg2-binary --quiet

import psycopg2
from google.colab import userdata

SUPABASE_URI = userdata.get('SUPABASE_URI')
conn = psycopg2.connect(SUPABASE_URI)
conn.autocommit = True
cur = conn.cursor()
print('✅ Conexión a Supabase establecida correctamente.')

# Verificar conteos
for tabla in ['orders','customers','order_items','reviews','products']:
    cur.execute(f'SELECT COUNT(*) FROM {tabla}')
    print(f'   {tabla:<20} {cur.fetchone()[0]:>10,} registros')

✅ Conexión a Supabase establecida correctamente.
   orders                   99,441 registros
   customers                99,441 registros
   order_items             112,650 registros
   reviews                  98,410 registros
   products                 32,951 registros


---
## Paso 2 — Función de análisis

Función auxiliar que ejecuta EXPLAIN ANALYZE y muestra el plan de ejecución de forma legible.

In [22]:
import re

def explain_query(label, sql, params=None):
    """Ejecuta EXPLAIN ANALYZE y muestra métricas clave."""
    print(f'\n{"="*60}')
    print(f' {label}')
    print(f'{"="*60}')

    explain_sql = f'EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {sql}'
    cur.execute(explain_sql, params)
    plan_rows = [r[0] for r in cur.fetchall()]
    plan_text = '\n'.join(plan_rows)

    # Mostrar plan completo
    print(plan_text)

    # Extraer métricas clave
    planning  = re.search(r'Planning Time: ([\d.]+) ms', plan_text)
    execution = re.search(r'Execution Time: ([\d.]+) ms', plan_text)
    seq_scans = len(re.findall(r'Seq Scan', plan_text))
    idx_scans = len(re.findall(r'Index Scan|Index Only Scan', plan_text))

    print(f'\n📊 MÉTRICAS:')
    print(f'   Planning Time:  {planning.group(1) if planning else "N/A"} ms')
    print(f'   Execution Time: {execution.group(1) if execution else "N/A"} ms')
    print(f'   Seq Scans:      {seq_scans}')
    print(f'   Index Scans:    {idx_scans}')
    if seq_scans > 0:
        print(f'   ⚠️  Seq Scan detectado — candidato para índice')
    return plan_text

print('✅ Función explain_query lista.')

✅ Función explain_query lista.


---
## Q11 — Pedidos por cliente con total pagado

**Patrón OLTP:** consulta frecuente en perfil de usuario. Involucra JOIN entre orders y order_payments filtrado por customer_id.

In [23]:
Q11 = """
SELECT
    o.id                   AS order_id,
    o.order_status,
    o.purchase_timestamp,
    SUM(op.payment_value)  AS total_paid
FROM orders         o
JOIN order_payments op ON op.order_id = o.id
WHERE o.customer_id = %s
GROUP BY o.id, o.order_status, o.purchase_timestamp
ORDER BY o.purchase_timestamp DESC
"""

plan_q11 = explain_query(
    'Q11 — Pedidos por cliente con total pagado',
    Q11,
    params=('2462fe3e-d4b7-4672-9144-c141174d00f3',)
)


 Q11 — Pedidos por cliente con total pagado
Sort  (cost=5.32..5.33 rows=1 width=65) (actual time=0.041..0.042 rows=1 loops=1)
  Sort Key: o.purchase_timestamp DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=8
  ->  GroupAggregate  (cost=5.29..5.31 rows=1 width=65) (actual time=0.036..0.037 rows=1 loops=1)
        Group Key: o.id
        Buffers: shared hit=8
        ->  Sort  (cost=5.29..5.29 rows=1 width=39) (actual time=0.029..0.029 rows=1 loops=1)
              Sort Key: o.id
              Sort Method: quicksort  Memory: 25kB
              Buffers: shared hit=8
              ->  Nested Loop  (cost=0.83..5.28 rows=1 width=39) (actual time=0.023..0.024 rows=1 loops=1)
                    Buffers: shared hit=8
                    ->  Index Scan using idx_orders_customer on orders o  (cost=0.42..2.64 rows=1 width=33) (actual time=0.011..0.012 rows=1 loops=1)
                          Index Cond: (customer_id = '2462fe3e-d4b7-4672-9144-c141174d00f3'::uuid)
            

---
## Q12 — Detalle completo de orden (4 JOINs)

**Patrón OLTP:** consulta de detalle de orden. Encadena 4 JOINs: order_items → products → categories → sellers.

In [24]:
# Obtener un order_id real
cur.execute('SELECT id FROM orders LIMIT 1')
order_id_real = cur.fetchone()[0]
print(f'order_id de prueba: {order_id_real}')

Q12 = """
SELECT
    oi.item_sequence,
    p.name                 AS product_name,
    c.name                 AS category,
    s.name                 AS seller,
    oi.price,
    oi.freight_value,
    oi.price + oi.freight_value AS subtotal
FROM order_items  oi
JOIN products     p  ON p.id = oi.product_id
JOIN categories   c  ON c.id = p.category_id
JOIN sellers      s  ON s.id = oi.seller_id
WHERE oi.order_id = %s
ORDER BY oi.item_sequence
"""

plan_q12 = explain_query(
    'Q12 — Detalle completo de orden (4 JOINs)',
    Q12,
    params=(order_id_real,)
)

order_id de prueba: 5aca8afe-9ae9-4e86-b0b5-c00c659e642c

 Q12 — Detalle completo de orden (4 JOINs)
Nested Loop  (cost=1.13..7.81 rows=1 width=98) (actual time=0.030..0.032 rows=1 loops=1)
  Buffers: shared hit=12
  ->  Nested Loop  (cost=0.85..5.31 rows=1 width=66) (actual time=0.023..0.024 rows=1 loops=1)
        Buffers: shared hit=9
        ->  Nested Loop  (cost=0.71..5.14 rows=1 width=65) (actual time=0.020..0.021 rows=1 loops=1)
              Buffers: shared hit=7
              ->  Index Scan using pk_order_items on order_items oi  (cost=0.42..2.64 rows=1 width=48) (actual time=0.010..0.011 rows=1 loops=1)
                    Index Cond: (order_id = '5aca8afe-9ae9-4e86-b0b5-c00c659e642c'::uuid)
                    Buffers: shared hit=4
              ->  Index Scan using products_pkey on products p  (cost=0.29..2.51 rows=1 width=49) (actual time=0.005..0.005 rows=1 loops=1)
                    Index Cond: (id = oi.product_id)
                    Buffers: shared hit=3
        -> 

---
## Q13 — Tiempo promedio de entrega por estado

**Patrón OLAP:** agregación compleja con 4 JOINs y cálculo de diferencia temporal. Alta carga computacional.

In [25]:
Q13 = """
SELECT
    g.state,
    AVG(
        EXTRACT(EPOCH FROM (o.delivered_customer_date - o.purchase_timestamp))
        / 86400
    )::NUMERIC(6,1) AS avg_delivery_days
FROM orders      o
JOIN order_items oi ON oi.order_id     = o.id
JOIN sellers      s  ON s.id            = oi.seller_id
JOIN geolocations g  ON g.id            = s.geolocation_id
WHERE o.delivered_customer_date IS NOT NULL
GROUP BY g.state
ORDER BY avg_delivery_days
"""

plan_q13 = explain_query('Q13 — Tiempo promedio de entrega por estado', Q13)


 Q13 — Tiempo promedio de entrega por estado
Sort  (cost=7878.50..7878.57 rows=27 width=17) (actual time=145.226..151.304 rows=21 loops=1)
  Sort Key: ((avg((EXTRACT(epoch FROM (o.delivered_customer_date - o.purchase_timestamp)) / '86400'::numeric)))::numeric(6,1))
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=3352
  ->  Finalize GroupAggregate  (cost=7874.15..7877.86 rows=27 width=17) (actual time=145.158..151.288 rows=21 loops=1)
        Group Key: g.state
        Buffers: shared hit=3352
        ->  Gather Merge  (cost=7874.15..7877.25 rows=27 width=35) (actual time=145.145..151.240 rows=42 loops=1)
              Workers Planned: 1
              Workers Launched: 1
              Buffers: shared hit=3352
              ->  Sort  (cost=6874.14..6874.20 rows=27 width=35) (actual time=135.528..135.533 rows=21 loops=2)
                    Sort Key: g.state
                    Sort Method: quicksort  Memory: 26kB
                    Buffers: shared hit=3352
                

---
## Q16 — Score promedio de reviews por categoría

**Patrón OLAP:** 5 JOINs encadenados con GROUP BY y agregación. Una de las consultas más costosas del sistema.

In [26]:
Q16 = """
SELECT
    c.name                     AS category,
    AVG(r.score)::NUMERIC(3,2) AS avg_score,
    COUNT(r.id)                AS total_reviews
FROM reviews      r
JOIN orders       o  ON o.id         = r.order_id
JOIN order_items  oi ON oi.order_id  = o.id
JOIN products     p  ON p.id         = oi.product_id
JOIN categories   c  ON c.id         = p.category_id
GROUP BY c.name
ORDER BY avg_score DESC
"""

plan_q16 = explain_query('Q16 — Score promedio de reviews por categoría (5 JOINs)', Q16)


 Q16 — Score promedio de reviews por categoría (5 JOINs)
Sort  (cost=10797.88..10798.22 rows=137 width=37) (actual time=198.548..261.491 rows=73 loops=1)
  Sort Key: ((avg(r.score))::numeric(3,2)) DESC
  Sort Method: quicksort  Memory: 28kB
  Buffers: shared hit=3901
  ->  Finalize GroupAggregate  (cost=10774.18..10793.02 rows=137 width=37) (actual time=198.401..261.452 rows=73 loops=1)
        Group Key: c.name
        Buffers: shared hit=3901
        ->  Gather Merge  (cost=10774.18..10789.93 rows=137 width=57) (actual time=198.388..261.365 rows=145 loops=1)
              Workers Planned: 1
              Workers Launched: 1
              Buffers: shared hit=3901
              ->  Sort  (cost=9774.17..9774.51 rows=137 width=57) (actual time=194.507..194.517 rows=72 loops=2)
                    Sort Key: c.name
                    Sort Method: quicksort  Memory: 30kB
                    Buffers: shared hit=3901
                    Worker 0:  Sort Method: quicksort  Memory: 30kB
      

---
## Q5 — Filtro JSONB con operador @>

**Tipo avanzado:** búsqueda dentro de JSONB usando el operador de contenencia. Sin índice GIN genera Seq Scan sobre product_details.

In [27]:
Q5 = """
SELECT
    p.name,
    pd.attributes->>'product_weight_g' AS weight_g
FROM products        p
JOIN product_details pd ON pd.product_id = p.id
WHERE pd.attributes @> '{"product_photos_qty": 4.0}'
"""

plan_q5 = explain_query('Q5 — Filtro JSONB con operador @> (sin índice GIN)', Q5)


 Q5 — Filtro JSONB con operador @> (sin índice GIN)
Hash Join  (cost=996.12..1760.14 rows=3203 width=49) (actual time=4.233..12.038 rows=2428 loops=1)
  Hash Cond: (p.id = pd.product_id)
  Buffers: shared hit=1237
  ->  Seq Scan on products p  (cost=0.00..669.51 rows=32951 width=33) (actual time=0.009..2.561 rows=32951 loops=1)
        Buffers: shared hit=340
  ->  Hash  (cost=956.08..956.08 rows=3203 width=187) (actual time=4.209..4.210 rows=2428 loops=1)
        Buckets: 4096  Batches: 1  Memory Usage: 555kB
        Buffers: shared hit=897
        ->  Bitmap Heap Scan on product_details pd  (cost=25.04..956.08 rows=3203 width=187) (actual time=0.833..3.591 rows=2428 loops=1)
              Recheck Cond: (attributes @> '{"product_photos_qty": 4.0}'::jsonb)
              Rows Removed by Index Recheck: 1055
              Heap Blocks: exact=881
              Buffers: shared hit=897
              ->  Bitmap Index Scan on idx_pdetails_attributes  (cost=0.00..24.24 rows=3203 width=0) (actua

---
## Q6 — Filtro JSONB con comparación numérica

**Tipo avanzado:** cast numérico desde JSONB en cláusula WHERE. Sin índice genera Seq Scan + cast por cada fila.

In [28]:
Q6 = """
SELECT
    p.name,
    (pd.attributes->>'product_weight_g')::float AS weight_g
FROM products        p
JOIN product_details pd ON pd.product_id = p.id
WHERE (pd.attributes->>'product_weight_g')::float > 1000
ORDER BY weight_g DESC
LIMIT 20
"""

plan_q6 = explain_query('Q6 — Filtro JSONB con cast numérico', Q6)


 Q6 — Filtro JSONB con cast numérico
Limit  (cost=2818.00..2818.05 rows=20 width=25) (actual time=34.395..34.401 rows=20 loops=1)
  Buffers: shared hit=1231
  ->  Sort  (cost=2818.00..2845.46 rows=10984 width=25) (actual time=34.394..34.397 rows=20 loops=1)
        Sort Key: (((pd.attributes ->> 'product_weight_g'::text))::double precision) DESC
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=1231
        ->  Hash Join  (cost=1687.32..2525.72 rows=10984 width=25) (actual time=16.432..31.767 rows=12892 loops=1)
              Hash Cond: (p.id = pd.product_id)
              Buffers: shared hit=1231
              ->  Seq Scan on products p  (cost=0.00..669.51 rows=32951 width=33) (actual time=0.009..2.845 rows=32951 loops=1)
                    Buffers: shared hit=340
              ->  Hash  (cost=1550.02..1550.02 rows=10984 width=187) (actual time=16.409..16.410 rows=12892 loops=1)
                    Buckets: 16384  Batches: 1  Memory Usage: 2898kB
        

---
## Q9 — Búsqueda fuzzy con pg_trgm

**Extensión pg_trgm:** búsqueda tolerante a errores tipográficos. Requiere la extensión instalada y un índice GIN con gin_trgm_ops.

In [29]:
# Activar extensión pg_trgm si no está activa
cur.execute('CREATE EXTENSION IF NOT EXISTS pg_trgm')
print('✅ Extensión pg_trgm activa.')

Q9 = """
SELECT
    p.name,
    similarity(p.name, 'Product') AS score
FROM products p
WHERE similarity(p.name, 'Product') > 0.3
ORDER BY score DESC
LIMIT 10
"""

plan_q9 = explain_query('Q9 — Búsqueda fuzzy con pg_trgm (sin índice trgm)', Q9)

✅ Extensión pg_trgm activa.

 Q9 — Búsqueda fuzzy con pg_trgm (sin índice trgm)
Limit  (cost=1099.09..1099.11 rows=10 width=21) (actual time=167.017..167.019 rows=10 loops=1)
  Buffers: shared hit=340
  ->  Sort  (cost=1099.09..1126.55 rows=10984 width=21) (actual time=167.015..167.016 rows=10 loops=1)
        Sort Key: (similarity((name)::text, 'Product'::text)) DESC
        Sort Method: top-N heapsort  Memory: 25kB
        Buffers: shared hit=340
        ->  Seq Scan on products p  (cost=0.00..861.73 rows=10984 width=21) (actual time=0.803..160.155 rows=32951 loops=1)
              Filter: (similarity((name)::text, 'Product'::text) > '0.3'::double precision)
              Buffers: shared hit=340
Planning Time: 0.090 ms
Execution Time: 167.053 ms

📊 MÉTRICAS:
   Planning Time:  0.090 ms
   Execution Time: 167.053 ms
   Seq Scans:      1
   Index Scans:    0
   ⚠️  Seq Scan detectado — candidato para índice


---
## Resumen de hallazgos

In [30]:
print('\n📋 RESUMEN DE HALLAZGOS — Consultas críticas Ecommify')
print('='*70)
print(f'{"Query":<8} {"Descripción":<40} {"Problema detectado"}')
print('-'*70)
hallazgos = [
    ('Q11', 'Pedidos por cliente',             'Seq Scan en orders sin índice por customer_id'),
    ('Q12', 'Detalle orden (4 JOINs)',          '4 JOINs encadenados — evaluar índices compuestos'),
    ('Q13', 'Tiempo entrega por estado',        'Aggregación sobre 100K+ filas sin partición'),
    ('Q16', 'Score reviews por categoría',      '5 JOINs — candidato para vista materializada'),
    ('Q5',  'JSONB @> sin GIN',                 'Seq Scan en product_details — necesita índice GIN'),
    ('Q6',  'JSONB cast numérico',              'Cast por fila — índice de expresión recomendado'),
    ('Q9',  'Búsqueda fuzzy pg_trgm',           'Seq Scan en products — necesita índice GIN trgm'),
]
for q, desc, problema in hallazgos:
    print(f'{q:<8} {desc:<40} {problema}')

print('\n🎯 PRIORIDAD DE OPTIMIZACIÓN:')
print('  1. Índice GIN en product_details.attributes   (Q5, Q6)')
print('  2. Índice GIN trgm en products.name            (Q9)')
print('  3. Índice B-tree en orders.customer_id         (Q11)')
print('  4. Vista materializada para Q16               (Q16)')
print('  5. Particionamiento orders por fecha           (Q13)')

cur.close()
conn.close()
print('\n✅ Análisis completo. Conexión cerrada.')


📋 RESUMEN DE HALLAZGOS — Consultas críticas Ecommify
Query    Descripción                              Problema detectado
----------------------------------------------------------------------
Q11      Pedidos por cliente                      Seq Scan en orders sin índice por customer_id
Q12      Detalle orden (4 JOINs)                  4 JOINs encadenados — evaluar índices compuestos
Q13      Tiempo entrega por estado                Aggregación sobre 100K+ filas sin partición
Q16      Score reviews por categoría              5 JOINs — candidato para vista materializada
Q5       JSONB @> sin GIN                         Seq Scan en product_details — necesita índice GIN
Q6       JSONB cast numérico                      Cast por fila — índice de expresión recomendado
Q9       Búsqueda fuzzy pg_trgm                   Seq Scan en products — necesita índice GIN trgm

🎯 PRIORIDAD DE OPTIMIZACIÓN:
  1. Índice GIN en product_details.attributes   (Q5, Q6)
  2. Índice GIN trgm en products.name  